# M1 — Market question and sources
## Step 1: Define the study before collecting data

This workshop builds **Uranium Market Signals Lab**. Start with the question and data contract so later choices can be checked against a written research design.

This notebook uses only Python's standard library. Run the code cells in order in a Python 3 notebook environment. If Jupyter is not installed, stop at that setup checkpoint before installing dependencies.

This step does **not** download data. Source links below are starting points from the specification; availability, holdings, and terms have not yet been verified.


## Research question and hypothesis

At each month-end, can historically available market signals identify eligible uranium-related equities that outperform URA over the following month?

**Testable hypothesis:** lagged momentum, volatility, relative strength, and monthly uranium-market context provide useful information beyond a majority-class baseline and a simple momentum rule. Improvement must be evaluated chronologically on unseen dates, then tested economically after transaction costs. A negative result is a valid outcome.

The label is `1` when the stock's next-month total return exceeds URA's next-month total return, otherwise `0`. Missing future returns remain missing, rather than becoming class `0`.

A separate descriptive study measures rolling 3- and 5-year nominal and real total returns for Cameco and a basket defined before inspecting performance. Overlapping windows are not independent observations.


In [ ]:
study = {
    "signal_frequency": "month_end",
    "primary_horizon_months": 1,
    "benchmark": "URA",
    "target": "stock_forward_total_return > benchmark_forward_total_return",
    "execution": "no earlier than the next tradable session after signal formation",
    "validation": "chronological walk-forward with a final untouched holdout",
    "universe_size_goal": (10, 25),
    "universe_status": "pending holdings collection and history/liquidity audit",
    "currency_policy": "use a common currency or report currencies separately",
    "long_horizons_months": (36, 60),
}

for key, value in study.items():
    print(f"{key}: {value}")


## Planned sources

ETF membership supplies candidates; it does not establish equal uranium exposure. Keep miners, physical uranium vehicles, utilities, and equipment/service companies distinguishable. Final security selection follows a data-availability audit.

The following register is a **plan**, not evidence of retrieval or permission to redistribute. Populate provider terms, retrieval timestamps, effective dates, units, currencies, and local snapshot paths when inspecting each source. Choose an equity-price provider during the next source-review step.


In [ ]:
source_plan = [
    ("URA", "ETF holdings", "Global X", "https://www.globalxetfs.com/funds/ura"),
    ("URNM", "ETF holdings", "Sprott", "https://sprottetfs.com/urnm-sprott-uranium-miners-etf"),
    ("HURA", "ETF holdings", "Global X Canada", "https://www.globalx.ca/product/HURA"),
    ("URNJ", "ETF holdings", "Sprott", "https://sprottetfs.com/urnj-sprott-junior-uranium-miners-etf"),
    ("NLR", "ETF holdings", "VanEck", "https://www.vaneck.com/us/en/investments/uranium-nuclear-energy-etf-nlr/holdings/"),
    ("NUKZ", "ETF holdings", "Range", "https://www.rangeetfs.com/nukz"),
    ("PURANUSDM", "Monthly uranium benchmark", "FRED / IMF", "https://fred.stlouisfed.org/series/PURANUSDM"),
]
source_register = [
    {
        "source_id": source_id,
        "dataset": dataset,
        "provider": provider,
        "source_url": url,
        "review_status": "not_reviewed",
        "retrieved_at_utc": None,
        "effective_date": None,
        "available_at": None,
        "units": None,
        "currency": None,
        "frequency": None,
        "terms_notes": None,
        "snapshot_path": None,
    }
    for source_id, dataset, provider, url in source_plan
]

for source in source_register:
    print(source["source_id"], source["review_status"], source["source_url"])
print("Still to select: adjusted equity/ETF prices, monthly CPI, broad equity benchmark.")


## Holdings data contract

Use one row per security per ETF snapshot. A stable internal security identifier distinguishes securities from provider-specific ticker strings. Retain the exchange, share class, and currency to prevent accidental merges of different listings.

Deduplicate into a security table later, preserving the many-to-many ETF mapping. Record exclusions with reasons. Current holdings used historically must be disclosed as a current-constituent study with survivorship and membership bias.


In [ ]:
holdings_fields = (
    "source_id", "snapshot_date", "retrieved_at_utc", "source_url",
    "security_id", "ticker", "exchange", "company_name", "share_class",
    "security_currency", "exposure_group", "weight_pct", "review_notes",
)
exposure_groups = (
    "uranium_producer", "uranium_developer_explorer", "physical_uranium",
    "nuclear_utility", "nuclear_equipment_services", "other_related",
)

print("Holdings CSV header:")
print(",".join(holdings_fields))
print("Allowed exposure groups:", ", ".join(exposure_groups))


## Verification checkpoint

Run the checks below. They validate this initial study plan, not the quality or completeness of any market data.

Before continuing, explain in your own words:

1. Why can a stock outperform URA while still losing money?
2. Why are today's ETF holdings insufficient to reconstruct historical ETF membership?
3. Why must a monthly uranium observation have an availability date as well as an observation date?
4. Why do monthly rolling five-year returns provide fewer independent outcomes than their row count suggests?

**Next bounded step:** review sponsor sources, record dated holdings and terms, and choose price/CPI providers. M1 remains incomplete until its holdings inventory, candidate universe, and reviewed source register exist. M2 downloads and price-quality work follow that checkpoint.


In [ ]:
assert study["primary_horizon_months"] == 1
assert study["benchmark"] == "URA"
assert len({row["source_id"] for row in source_register}) == len(source_register)
assert all(row["source_url"].startswith("https://") for row in source_register)
assert all(row["retrieved_at_utc"] is None for row in source_register)
assert len(holdings_fields) == len(set(holdings_fields))
assert len(exposure_groups) == 6
print("Step 1 plan checks passed. Source review and holdings collection are pending.")


## Step 2 — Inspect and verify the first holdings snapshots

The collection contains complete downloadable CSVs from [URNM](https://sprottetfs.com/urnm-sprott-uranium-miners-etf) and [URNJ](https://sprottetfs.com/urnj-sprott-junior-uranium-miners-etf), plus holdings extracted from the embedded page payload at [NUKZ](https://www.rangeetfs.com/nukz). All three are effective **2026-11-19**; retrieval timestamps are recorded separately in `data/reference/m1_source_review.json`. These are current snapshots, not historical membership records.

Raw downloads stay in ignored `data/raw/holdings/` because redistribution rights have not been established. The manifest records their hashes so a future run can verify the exact local inputs. A missing snapshot should fail clearly; downloading today's holdings would not reproduce this snapshot.

URA returned 429, HURA returned 403, and NLR redirected in a loop. These three are omitted from initial candidate discovery with documented reasons. NUKZ was subsequently recovered from its embedded page payload. HURA's visible top-ten list is not treated as a complete portfolio. The manifest separates metadata review from successful acquisition.


In [ ]:
from pathlib import Path
import csv
import hashlib
import io
import json
from collections import Counter

# Support launching from the project root or notebooks/.
project_root = Path.cwd()
if not (project_root / "AGENTS.md").is_file():
    project_root = project_root.parent
assert (project_root / "AGENTS.md").is_file(), "Run from the project root or notebooks/"
review = json.loads((project_root / "data/reference/m1_source_review.json").read_text())

holdings_by_fund = {}
for source in review["sources"]:
    if source["source_id"] not in {"URNM", "URNJ", "NUKZ"} or not source.get("snapshot_path"):
        print(source["source_id"], source["review_status"])
        continue
    snapshot = project_root / source["snapshot_path"]
    if not snapshot.is_file():
        raise FileNotFoundError(f"Original local snapshot is missing: {snapshot}")
    payload = snapshot.read_bytes()
    assert hashlib.sha256(payload).hexdigest() == source["sha256"], "Snapshot changed"
    rows = list(csv.DictReader(io.StringIO(payload.decode("utf-8-sig"))))
    assert len(rows) == source["row_count"]
    if source["source_id"] == "NUKZ":
        rows = [{"Security": row["description"], "Symbol": row["ticker"],
                 "SEDOL": "", "FIGI": row["figi"],
                 "Weight": row["percent_of_nav"].rstrip("%")}
                for row in rows]
    weight_sum = sum(float(row["Weight"]) for row in rows)
    assert abs(weight_sum - 100) < 1, "Investigate incomplete/incorrect weights"
    holdings_by_fund[source["source_id"]] = rows
    print(source["source_id"], source["effective_date"], len(rows), "rows; weight %", round(weight_sum, 2))


### Inspect identifiers before merging

A SEDOL identifies a security/listing rather than an entire company. Two listings of the same issuer can have different SEDOLs. The check below measures exact security overlap only; it does not deduplicate issuers or certify investability. Cash and missing identifiers stay visible for manual review. Exchange, share class, exposure category, and trading currency still require verification.


In [ ]:
for fund, rows in holdings_by_fund.items():
    sedols = [row["SEDOL"] for row in rows if row["SEDOL"]]
    duplicates = [key for key, count in Counter(sedols).items() if count > 1]
    print(fund, "duplicate SEDOLs:", duplicates)
    print(fund, "rows needing identifier review:",
          [(row["Security"], row["Symbol"]) for row in rows if not row["SEDOL"] or not row["Symbol"]])

security_sets = {
    fund: {row["SEDOL"] for row in rows if row["SEDOL"]}
    for fund, rows in holdings_by_fund.items()
}
assert {"URNM", "URNJ", "NUKZ"} == set(security_sets)
print("Exact shared SEDOLs:", len(security_sets["URNM"] & security_sets["URNJ"]))
print("Distinct nonempty SEDOLs:", len(security_sets["URNM"] | security_sets["URNJ"]))


### Macro sources and next checkpoint

- [FRED PURANUSDM](https://fred.stlouisfed.org/series/PURANUSDM) is a monthly IMF benchmark in nominal USD per pound, representing period averages. It is not an executable spot quote. The series carries an IMF copyright notice; the [FRED terms](https://fred.stlouisfed.org/legal/) require attention to third-party rights. No macro values were downloaded or redistributed here.
- Proposed deflator: [CPIAUCNS](https://fred.stlouisfed.org/series/CPIAUCNS), monthly US CPI-U, **not seasonally adjusted**, index 1982–84=100. This supports a US-dollar purchasing-power study; Canadian-dollar results would require separate currency/deflator treatment. Observation month and publication date are different.
- [Alpha Vantage documentation](https://www.alphavantage.co/documentation/) was inspected as one equity-price candidate. The selected first provider is now Yahoo via yfinance for personal educational research; Alpha Vantage is retained only as an alternative considered during review.

**Checkpoint:** run the two cells above and inspect the identifier warnings. Explain why the holdings dates and retrieval times differ, and why distinct SEDOL counts do not equal distinct company counts.

**M1 continuation:** initial candidate discovery uses the three accessible snapshots; the other sponsors have documented access exclusions. Sixteen USD listings and their dated ETF mappings are recorded below. M2 verifies provider currency, history, and date-specific liquidity; the final workflow uses those rules without selecting on returns.


## Steps 3–6 — Initial research universe and source decisions

The source omissions above are accepted limitations of this first version. They do not mean the funds lack relevant holdings. The 16-candidate universe includes miners, utilities, and equipment/services, using USD listings to avoid silent FX mixing. The `ticker_to_etf.csv` mapping records observed snapshot memberships only; it does not reconstruct earlier memberships.

The long-run basket is fixed in configuration before inspecting returns: CCJ, DNN, UEC, UUUU, URG, NXE. Broad exposure groups are manually assigned for research, not time-varying historical business classifications. Price-provider metadata verifies currency and listing identity in M2.

Provider selection: Yahoo via [yfinance](https://ranaroussi.github.io/yfinance/) for local personal educational prices; FRED's official CSV downloads for [PURANUSDM](https://fred.stlouisfed.org/series/PURANUSDM) and [CPIAUCNS](https://fred.stlouisfed.org/series/CPIAUCNS). Raw inputs and derived reports stay local; IMF redistribution is not cleared. Use SPY for broad equity comparison, URA as primary benchmark, and URNM where its history permits.


In [ ]:
import csv
with (project_root / "data/reference/universe.csv").open() as stream:
    candidate_universe = list(csv.DictReader(stream))
with (project_root / "data/reference/ticker_to_etf.csv").open() as stream:
    memberships = list(csv.DictReader(stream))
assert len({row["ticker"] for row in candidate_universe}) == 16
assert all(row["currency"] == "USD" for row in candidate_universe)
assert {row["ticker"] for row in memberships} == {row["ticker"] for row in candidate_universe}
for row in candidate_universe:
    print(row["ticker"], row["exposure_group"], row["source_etfs"])
print("Initial universe checkpoint passed. Continue with M2 history/currency/liquidity verification.")
